# Lecture 14: Hypothesis Testing 3 — $p$-values, Confidence Regions, Test-CI Duality

**Data 145, Spring 2026: Evidence and Uncertainty**
**Instructors:** Ani Adhikari, William Fithian

---

**Please run all the code cells below before you start reading.**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm, binom, expon, gamma
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.style.use('fivethirtyeight')
%matplotlib inline

# Color scheme
COLOR_DATA = 'steelblue'
COLOR_APPROX = 'firebrick'

# Reproducibility
rng = np.random.default_rng(145)

print("Setup complete.")

---

## 1. $p$-values Formalized

### Why Move Beyond Reject/Accept?

So far: given a significance level $\alpha$, we either reject or fail to reject $H_0$. But in practice, we usually want to know *how much* evidence there is against $H_0$. Would we reject at $\alpha = 0.05$? What about $0.01$? $0.001$?

The **$p$-value** answers all these questions at once. If we reject for large $T(X)$, the $p$-value is:

$$p(x) = P_{H_0}\bigl(T(X) \geq T(x)\bigr)$$

In words: how "extreme" is our observed statistic under the null? This is exactly what `scipy.stats.kstest` computed for us in Lecture 11 — now we formalize it.

### Formal Definition (Composite Nulls)

For a simple null, the $p$-value is straightforward. But what if $H_0: \theta \in \Theta_0$ is composite? Different null values $\theta \in \Theta_0$ give different probabilities. We take the **worst case**:

$$p(x) = \sup_{\theta \in \Theta_0} P_\theta\bigl(T(X) \geq T(x)\bigr)$$

This ensures the $p$-value is valid for all null parameter values simultaneously.

### Examples

- **Binomial**: $X \sim \text{Binom}(n, p)$, test $H_0: p \leq 0.5$ vs $H_1: p > 0.5$. Since $P_p(X \geq x)$ is increasing in $p$ for fixed $x$, the supremum is attained at the boundary: $p(x) = P_{0.5}(X \geq x)$.

- **$z$-test (two-sided)**: $X \sim N(\theta, 1)$, test $H_0: \theta = 0$ vs $H_1: \theta \neq 0$. Reject for large $|X|$:
$$p(x) = P_0(|X| \geq |x|) = 2\bigl(1 - \Phi(|x|)\bigr)$$

### Super-Uniformity

The key property of $p$-values:

$$P_\theta\bigl(p(X) \leq \alpha\bigr) \leq \alpha \quad \text{for all } \theta \in \Theta_0$$

**Intuition.** Under the null, the $p$-value is "spread out" over $[0, 1]$ — at most $\alpha$ of the probability mass falls below $\alpha$.

**Consequence.** Rejecting when $p(X) \leq \alpha$ gives a valid level-$\alpha$ test. The $p$-value is the smallest $\alpha$ at which we would reject — it converts a single observed dataset into a summary of evidence against $H_0$.

*You will prove super-uniformity on this week's worksheet.* For now, let's verify it by simulation.

In [ ]:
# Figure 1 (interactive): Z-statistic and p-value distributions under H0 vs H1
n_simulations = 10_000
z_null = rng.standard_normal(n_simulations)  # fixed draws, shifted by theta

def plot_pval_distribution(theta=0.0):
    z_samples = z_null + theta
    p_values = 2 * (1 - norm.cdf(np.abs(z_samples)))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # Left: Z-statistic histogram with null and actual densities
    axes[0].hist(z_samples, bins=40, density=True, color=COLOR_DATA, alpha=0.5,
                 edgecolor='white', label='Simulated $Z$')
    z_grid = np.linspace(-4, 8, 300)
    axes[0].plot(z_grid, norm.pdf(z_grid), color='black', linestyle='--',
                 linewidth=1.5, label='$N(0,1)$ (null)')
    if theta != 0:
        axes[0].plot(z_grid, norm.pdf(z_grid, loc=theta), color=COLOR_APPROX,
                     linewidth=1.5, label=f'$N({theta:.2f},1)$ (actual)')
    axes[0].set_xlabel('$Z$', fontsize=12)
    axes[0].set_ylabel('Density', fontsize=12)
    axes[0].set_title('$Z$-statistic distribution', fontsize=13, fontweight='bold')
    axes[0].legend(fontsize=9, loc='upper right')
    axes[0].set_xlim(-4, max(4, theta + 4))
    ymax_z = axes[0].get_ylim()[1]
    axes[0].set_ylim(0, ymax_z * 1.2)

    # Middle: histogram of p-values
    axes[1].hist(p_values, bins=20, density=True, color=COLOR_DATA, alpha=0.6,
                 edgecolor='white', label='Simulated $p$-values')
    axes[1].axhline(1, color='black', linestyle='--', linewidth=1.5,
                    label='$\\mathrm{Unif}(0,1)$')
    axes[1].set_xlabel('$p$-value', fontsize=12)
    axes[1].set_ylabel('Density', fontsize=12)
    if theta == 0:
        title_m = '$p$-values under $H_0$: $\\theta = 0$'
    else:
        title_m = f'$p$-values under $H_1$: $\\theta = {theta:.2f}$'
    axes[1].set_title(title_m, fontsize=13, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].set_xlim(0, 1)
    ymax_p = axes[1].get_ylim()[1]
    axes[1].set_ylim(0, ymax_p * 1.2)

    # Right: empirical CDF vs Unif(0,1)
    p_sorted = np.sort(p_values)
    ecdf_vals = np.arange(1, n_simulations + 1) / n_simulations
    axes[2].plot(p_sorted, ecdf_vals, color=COLOR_DATA, linewidth=2,
                 label='Empirical CDF')
    axes[2].plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=1.5,
                 label='$\\mathrm{Unif}(0,1)$ CDF')
    axes[2].set_xlabel('$\\alpha$', fontsize=12)
    axes[2].set_ylabel('$P(p \\leq \\alpha)$', fontsize=12)
    axes[2].set_title('Super-uniformity check', fontsize=13, fontweight='bold')
    axes[2].legend(fontsize=9)

    plt.tight_layout()
    plt.show()

theta_slider = widgets.FloatSlider(
    value=0.0, min=0.0, max=4.0, step=0.25,
    description='True mean \u03b8:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

widgets.interact(plot_pval_distribution, theta=theta_slider);

*__Figure 1 (interactive).__ The two-sided $z$-test in action. **Left:** histogram of simulated $Z$-statistics with the null $N(0,1)$ density (black dashed) and the actual $N(\theta,1)$ density overlaid. **Middle:** the corresponding $p$-value histogram. At $\theta = 0$ (under $H_0$), $p$-values are uniform on $[0,1]$; as $\theta$ increases (under $H_1$), they concentrate near zero. **Right:** the empirical CDF of $p$-values. Under $H_0$ it lies on the $45°$ line; under $H_1$ it rises steeply above the diagonal, indicating that $P(p \leq \alpha) > \alpha$ — the test has high power.*

### The $p$-value Depends on the Test

Different tests of the same null hypothesis can give very different $p$-values. Recall Lecture 13: when the alternative is multidirectional, no single test dominates. The choice of test statistic determines what kinds of departures from $H_0$ the $p$-value is sensitive to.

**Important:** The $p$-value is **not** "the probability that $H_0$ is true." It is the probability of seeing data as extreme as ours, *under the assumption that $H_0$ is true*. Confusing these is sometimes called the **prosecutor's fallacy**.

**Example (medical screening).** A screening test for a rare disease has a $5\%$ false positive rate and $100\%$ sensitivity (it never misses a true case). The disease affects $1$ in $1{,}000$ people. A patient tests positive.

The $p$-value analog here is $P(+ \mid \text{no disease}) = 0.05$ — small enough to "reject the null" at the $5\%$ level. But what we really want is $P(\text{disease} \mid +)$. By Bayes' theorem:

$$P(\text{disease} \mid +) = \frac{P(+ \mid \text{disease}) \cdot P(\text{disease})}{P(+)} = \frac{1 \times 0.001}{1 \times 0.001 + 0.05 \times 0.999} \approx 0.02$$

Despite the "significant" positive test, there is only about a $2\%$ chance the patient actually has the disease — because the disease is so rare (the prior $P(\text{disease}) = 0.001$ is very small). The false positive rate ($p$-value) is not the probability that $H_0$ is true; to get that, you need a prior.

---

## 2. Confidence Regions

### Motivation: Coin Flipping

A $p$-value tells us about the evidence against a *specific* null. But we often want to know: **what range of parameter values is consistent with the data?**

Consider two coin-flipping experiments, both testing $H_0: p = 0.5$:

- **Scenario A**: $n = 50$ flips, 29 heads ($\hat{p} = 0.58$). Somewhat far from $0.5$, but small $n$.
- **Scenario B**: $n = 5{,}000$ flips, 2600 heads ($\hat{p} = 0.52$). Very close to $0.5$, but huge $n$.

Both distance from $\theta_0$ and sample size matter. Which gives stronger evidence against the null? Let's compute both $p$-values and confidence intervals:

In [ ]:
# Coin-flipping: sample size vs effect size
# Scenario A: n=50, 29 heads (p_hat=0.58) — wide CI includes 0.5
# Scenario B: n=5000, 2600 heads (p_hat=0.52) — narrow CI excludes 0.5
scenarios = [
    {'n': 50,   'k': 29,   'label': 'A'},
    {'n': 5000, 'k': 2600, 'label': 'B'},
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, s in zip(axes, scenarios):
    n, k = s['n'], s['k']
    p_hat = k / n
    # Two-sided exact binomial p-value
    p_val = 2 * min(binom.cdf(k, n, 0.5), 1 - binom.cdf(k - 1, n, 0.5))
    p_val = min(p_val, 1.0)
    # Normal approximation 95% CI
    se = np.sqrt(p_hat * (1 - p_hat) / n)
    ci_lo = p_hat - 1.96 * se
    ci_hi = p_hat + 1.96 * se

    # Plot: point estimate with CI
    ax.errorbar(p_hat, 0.5, xerr=[[p_hat - ci_lo], [ci_hi - p_hat]],
                fmt='o', color=COLOR_DATA, markersize=10, capsize=8,
                elinewidth=2.5, capthick=2, label=f'$\\hat{{p}} = {p_hat:.2f}$')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1.5, label='$H_0: p = 0.5$')

    ax.set_xlim(0.35, 0.75)
    ax.set_ylim(-0.5, 1.5)
    ax.set_yticks([])
    ax.set_xlabel('$p$', fontsize=12)
    ax.set_title(f'Scenario {s["label"]}: $n = {n:,}$, ${k}$ heads\n'
                 f'$p$-value $= {p_val:.4f}$, 95% CI $= [{ci_lo:.3f},\\, {ci_hi:.3f}]$',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.show()

*__Figure 2.__ Two coin-flipping experiments testing $H_0: p = 0.5$. Scenario A ($n = 50$, $\hat{p} = 0.58$) has a moderate effect size but small sample: the wide CI includes $0.5$, so we fail to reject — there isn't enough data to be sure the coin is biased. Scenario B ($n = 5{,}000$, $\hat{p} = 0.52$) has a tiny effect but huge sample: the narrow CI excludes $0.5$, so we reject $H_0$ — high precision detects even a small departure. A confidence interval conveys both the magnitude and precision of an estimate, information a $p$-value alone cannot capture.*

### Definition

Recall the estimation problem from Lectures 3–5: given data $X$, we want to estimate some quantity of interest $g(\theta)$. A point estimate $T(X)$ gives a single best guess, but how precise is it? A **confidence region** gives a set of plausible values.

**Definition.** A **$1 - \alpha$ confidence region** for $g(\theta)$ is a set-valued function $C(X)$ satisfying:

$$P_\theta\bigl(C(X) \ni g(\theta)\bigr) \geq 1 - \alpha \quad \text{for all } \theta$$

Note the deliberate use of "$\ni$" (contains) rather than "$\in$" (is in). Think **"subject, verb, object"**:

- $C(X)$ is the **subject** — active, random, different each time we collect data
- $\ni$ is the **verb** — "covers"
- $g(\theta)$ is the **object** — fixed, acted upon, the same true value every time

The confidence region $C(X)$ is the random object. The parameter $g(\theta)$ is fixed. Before we collect data, there is a $(1 - \alpha)$ probability that the random $C(X)$ will land in a position that covers the fixed $g(\theta)$.

### Correct Interpretation

- ✓ **Correct**: "Before we collect data, there is a $95\%$ chance that $C(X)$ will cover $g(\theta)$"
- ✗ **Incorrect**: "There is a $95\%$ chance that $g(\theta)$ is in $[0.8, 1.1]$" (after computing the CI)

Once $C(X)$ is realized — say, as $[0.8, 1.1]$ — it either covers $g(\theta)$ or it doesn't. There is no more randomness. The $95\%$ is a statement about the **procedure**, not about any one particular interval.

This is a **frequentist** guarantee: if we repeat the experiment many times, about $95\%$ of our CIs will cover the true $g(\theta)$. If you want a probability statement about $\theta$ itself, you need a Bayesian **credible interval** (Lectures 6–8), which requires a prior.

### Confidence Intervals and Confidence Bounds

When $g(\theta) \in \mathbb{R}$, common types of confidence regions:

- $C(X) = [C_L(X), C_U(X)]$: a **confidence interval** (CI)
- $C(X) = [C_L(X), \infty)$: a **lower confidence bound** (LCB)
- $C(X) = (-\infty, C_U(X)]$: an **upper confidence bound** (UCB)

Typically: confidence intervals come from two-sided tests; confidence bounds from one-sided tests.

### Example: Exponential Confidence Interval

$X \sim \text{Exp}(\theta)$, where $\theta > 0$ is the mean. We want a confidence interval for $\theta$.

**Lower bound from UMP test.** The UMP test of $H_0: \theta \leq \theta_0$ rejects for large $X$. The threshold is $c_\alpha = -\theta_0 \log(\alpha)$ (since $P_{\theta_0}(X \geq c) = e^{-c/\theta_0}$). We accept $H_0$ when $X < -\theta_0 \log(\alpha)$, which (rearranging) is equivalent to $\theta_0 > X / (-\log \alpha)$.

Inverting: we **fail to reject** $H_0: \theta \leq \theta_0$ whenever $\theta_0 \geq X / (-\log \alpha)$, giving a lower confidence bound:

$$C_L(X) = \frac{X}{-\log \alpha}$$

**Upper bound from the other direction.** The UMP test of $H_0: \theta \geq \theta_0$ rejects for small $X$. We accept when $X > -\theta_0 \log(1 - \alpha)$, i.e., when $\theta_0 < X / (-\log(1 - \alpha))$. Combining both directions:

$$C(X) = \left[\frac{X}{-\log(\alpha/2)},\; \frac{X}{-\log(1 - \alpha/2)}\right]$$

This is a $1 - \alpha$ confidence interval for $\theta$. The same idea generalizes to $n$ observations using the $\text{Gamma}(n, \theta)$ distribution of $T = \sum X_i$; we'll visualize this below.

---

## 3. Duality of Tests and Confidence Regions

### Test $\to$ Confidence Region

Suppose for every value $a$, we have a level-$\alpha$ test $\phi(X; a)$ of $H_0: g(\theta) = a$ vs $H_1: g(\theta) \neq a$.

Define:

$$C(X) = \{a : \phi(X; a) < 1\}$$

This is the set of all null values that are **not rejected** by the data.

**Claim:** $C(X)$ is a valid $1 - \alpha$ confidence region.

**Proof:**
$$P_\theta\bigl(C(X) \ni g(\theta)\bigr) = P_\theta\bigl(\phi(X;\, g(\theta)) < 1\bigr) \geq 1 - \alpha$$

The last inequality holds because $\phi$ is a level-$\alpha$ test, so $E_\theta[\phi(X;\, g(\theta))] \leq \alpha$.

This construction is called **inverting a family of tests**.

### Confidence Region $\to$ Test

The duality goes both ways. Given a $1 - \alpha$ confidence region $C(X)$, define:

$$\phi(x; a) = \mathbf{1}\{a \notin C(x)\}$$

**Claim:** This is a valid level-$\alpha$ test of $H_0: g(\theta) = a$.

**Proof:**
$$E_\theta[\phi(X;\, g(\theta))] = P_\theta\bigl(g(\theta) \notin C(X)\bigr) \leq \alpha$$

### The Duality in Words

The confidence interval is exactly the set of parameter values that **"the data don't reject."** A value is inside the CI if and only if the corresponding test would not reject at level $\alpha$.

### Connections to Earlier Material

This duality is not new — you've been using it all along!

- **Asymptotic normal CI from Lectures 4–5**: The CI $\hat\theta \pm z_{\alpha/2} \cdot \text{SE}$ is exactly the set of $\theta_0$ values for which the $z$-test of $H_0: \theta = \theta_0$ would not reject. This is test inversion.

- **$t$-test CI from Lecture 13**: $\bar{X} \pm t_{\alpha/2, n-1} \cdot S/\sqrt{n}$ is the inversion of the $t$-test of $H_0: \mu = \mu_0$. The set of $\mu_0$ values that produce $|T| \leq t_{\alpha/2, n-1}$ is exactly this interval.

In both cases, the CI is the set of null values "consistent with the data" at level $\alpha$.

### Visualizing the Duality

For a concrete illustration, consider $n = 5$ iid observations from $\text{Exp}(\theta)$. The sufficient statistic $T = \sum_{i=1}^5 X_i \sim \text{Gamma}(5, \theta)$, and the ratio $T/\theta \sim \text{Gamma}(5, 1)$ is a pivotal quantity. Using the quantiles of $\text{Gamma}(5, 1)$:

$$C(\mathbf{X}) = \left[\frac{T}{q_{1-\alpha/2}},\; \frac{T}{q_{\alpha/2}}\right]$$

where $q_p$ denotes the $p$-th quantile. First, we draw many samples and compute CIs to illustrate the **randomness** of $C(X)$. Then, for a single sample, we show how the CI arises from inverting a family of tests.

In [ ]:
# Repeated confidence intervals: n=5 iid Exp(theta=2), 95% CI via Gamma
theta_true = 2.0
alpha_ci = 0.05
n_obs = 5
n_samples = 50

# Gamma(n, 1) quantiles for the CI
q_lo = gamma.ppf(alpha_ci / 2, n_obs)       # alpha/2 quantile
q_hi = gamma.ppf(1 - alpha_ci / 2, n_obs)   # 1-alpha/2 quantile

fig, ax = plt.subplots(figsize=(10, 7))

n_covers = 0
for i in range(n_samples):
    x_vals = rng.exponential(scale=theta_true, size=n_obs)
    t = x_vals.sum()
    x_bar = t / n_obs
    ci_lo = t / q_hi
    ci_hi = t / q_lo
    covers = ci_lo <= theta_true <= ci_hi
    n_covers += covers
    color = COLOR_DATA if covers else COLOR_APPROX
    lw = 1.5 if covers else 2.5
    ax.plot([ci_lo, ci_hi], [i, i], color=color, linewidth=lw,
            solid_capstyle='round')
    ax.plot(x_bar, i, 'o', color=color, markersize=4)

# True parameter
ax.axvline(theta_true, color='black', linestyle='--', linewidth=1.5,
           label=f'True $\\theta = {theta_true}$')

ax.set_xlabel('$\\theta$', fontsize=12)
ax.set_ylabel('Sample number', fontsize=12)
ax.set_title(f'Repeated 95% CIs for $\\theta$ ($n = {n_obs}$): {n_covers} of {n_samples} cover',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.set_yticks([0, 9, 19, 29, 39, 49])
ax.set_yticklabels([1, 10, 20, 30, 40, 50])
plt.tight_layout()
plt.show()

*__Figure 3.__ Fifty 95% confidence intervals for the mean $\theta$ of an $\text{Exp}(\theta = 2)$ distribution, each computed from $n = 5$ iid observations. The CI is based on the pivotal quantity $T/\theta \sim \text{Gamma}(5, 1)$, where $T = \sum_{i=1}^5 X_i$. Blue intervals cover the true $\theta = 2$ (dashed line); red intervals miss. The dots mark the sample means $\bar{X}$. About 95% of intervals cover — this is the frequentist guarantee.*

In [ ]:
# Interactive test-CI duality: n=5 iid Exp(theta=2)
# Plot p-value as a function of theta_0; CI = {theta_0 : p(theta_0) > alpha}
theta_true_dual = 2.0
alpha_dual = 0.05
n_obs_dual = 5
output = widgets.Output()

# Gamma(n, 1) quantiles for the CI
q_lo_dual = gamma.ppf(alpha_dual / 2, n_obs_dual)
q_hi_dual = gamma.ppf(1 - alpha_dual / 2, n_obs_dual)

def draw_duality(t_obs=None):
    # Draw the duality plot for n exponential observations.
    if t_obs is None:
        x_vals = rng.exponential(scale=theta_true_dual, size=n_obs_dual)
        t_obs = x_vals.sum()

    x_bar = t_obs / n_obs_dual

    # p-value for testing H0: theta = theta_0 (two-sided)
    # T ~ Gamma(n, theta_0)
    theta_grid = np.linspace(0.2, 8, 500)
    p_upper = gamma.sf(t_obs, n_obs_dual, scale=theta_grid)    # P(T >= t | theta_0)
    p_lower = gamma.cdf(t_obs, n_obs_dual, scale=theta_grid)   # P(T <= t | theta_0)
    p_vals = 2 * np.minimum(p_upper, p_lower)

    # CI from the Gamma pivotal quantity
    ci_lo = t_obs / q_hi_dual
    ci_hi = t_obs / q_lo_dual

    fig, ax = plt.subplots(figsize=(10, 5))

    # Shade the CI region
    in_ci = p_vals > alpha_dual
    ax.fill_between(theta_grid, 0, p_vals, where=in_ci,
                    color=COLOR_DATA, alpha=0.15, label='CI (fail to reject)')
    ax.fill_between(theta_grid, 0, p_vals, where=~in_ci,
                    color=COLOR_APPROX, alpha=0.1, label='Reject')

    # p-value curve
    ax.plot(theta_grid, p_vals, color=COLOR_DATA, linewidth=2.5)

    # alpha threshold
    ax.axhline(alpha_dual, color='gray', linestyle=':', linewidth=1.5,
               label=f'$\\alpha = {alpha_dual}$')

    # True theta and CI endpoints
    ax.axvline(theta_true_dual, color='black', linestyle='--', linewidth=1.5,
               label=f'True $\\theta = {theta_true_dual}$')
    ax.axvline(ci_lo, color=COLOR_APPROX, linestyle='--', linewidth=1, alpha=0.7)
    ax.axvline(ci_hi, color=COLOR_APPROX, linestyle='--', linewidth=1, alpha=0.7)

    covers = ci_lo <= theta_true_dual <= ci_hi
    status = "covers" if covers else "misses"
    ax.set_xlabel('$\\theta_0$ (null hypothesis value)', fontsize=12)
    ax.set_ylabel('$p$-value', fontsize=12)
    ax.set_title(f'Test-CI duality ($n = {n_obs_dual}$): $\\bar{{X}} = {x_bar:.2f}$, '
                 f'CI $= [{ci_lo:.2f},\\, {ci_hi:.2f}]$ ({status} true $\\theta$)',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(0, 8)
    plt.tight_layout()
    plt.show()

# Button to draw a new sample
button = widgets.Button(description='New Sample', button_style='info',
                        layout=widgets.Layout(width='150px'))

def on_click(b):
    with output:
        clear_output(wait=True)
        draw_duality()

button.on_click(on_click)

# Initial draw
with output:
    draw_duality()

display(button, output)

*__Figure 4 (interactive).__ Test-CI duality for $n = 5$ iid observations from $\text{Exp}(\theta = 2)$. The curve shows the two-sided $p$-value (based on the $\text{Gamma}(5, \theta_0)$ distribution of $T = \sum X_i$) as a function of the null value $\theta_0$. The 95% confidence interval (blue shading) is exactly the set of $\theta_0$ values where $p(\theta_0) > \alpha = 0.05$. Click "New Sample" to draw a fresh sample and see how the CI changes — the interval is random, but the true $\theta = 2$ (dashed) stays fixed.*

---

## 4. (Mis-)interpreting Hypothesis Tests

### Common Errors

1. **"$p < 0.05$, therefore there is an effect."** Confusion of statistical significance with practical significance. A statistically significant result might correspond to a tiny, scientifically meaningless effect.

2. **"$p > 0.05$, therefore there is no effect."** Absence of evidence is not evidence of absence. We may simply lack the sample size to detect a real effect.

3. **"$p < 10^{-6}$, therefore the effect is huge."** With a large enough sample, even a minuscule effect can produce a tiny $p$-value. The $p$-value reflects both effect size *and* sample size — it does not measure magnitude.

4. **"CI for men is $[0.2, 3.2]$, CI for women is $[-0.2, 2.8]$, therefore there's an effect for men but not women."** The CIs overlap substantially; the difference between "significant" and "not significant" is itself not necessarily significant.

5. **CIs are more informative than $p$-values alone**, because they convey the magnitude and precision of an estimate, not just whether it's nonzero.

### Conceptual Objections to Hypothesis Testing

- **"Point nulls are unrealistic."** In practice, $\theta$ is never exactly $0$. But: (1) we can test $|\theta| \leq \delta$ instead; (2) the two-sided test also tells us the *sign* of $\theta$; (3) we can invert to get a CI, which is often the more useful summary.

- **"Frequentist methods answer the wrong question."** Scientists want $P(H_0 \mid \text{data})$, not $P(\text{data} \mid H_0)$. This is a real limitation — recall Lectures 6–8, where Bayesians compute the posterior directly. But frequentist methods require fewer assumptions: no prior needed.

What additional information would a Bayesian need to compute $P(H_0 \mid \text{data})$? A prior probability for $H_0$ and a prior distribution over $\theta$ under $H_1$.

---

## 5. Summary and Next Time

### The Big Picture

Today we gave three complementary ways to summarize the evidence in data:

1. **$p$-values** summarize evidence against $H_0$. They are super-uniform under the null: $P_\theta(p(X) \leq \alpha) \leq \alpha$. Rejecting when $p \leq \alpha$ is a valid level-$\alpha$ test. But $p$-values depend on the choice of test and are **not** the probability that $H_0$ is true.

2. **Confidence regions** give the set of parameter values consistent with the data at level $1 - \alpha$. The CI $C(X)$ is the random object (subject); the parameter $g(\theta)$ is fixed (object). About $1 - \alpha$ of the time, $C(X)$ covers $g(\theta)$.

3. **Test-CI duality** ties these together: the CI is exactly the set of null values that the data do not reject. Inverting a family of tests gives a CI; turning a CI into a test gives back the family of tests.

### Key Takeaways

1. The $p$-value is super-uniform under $H_0$: $P_\theta(p(X) \leq \alpha) \leq \alpha$. Rejecting when $p \leq \alpha$ gives a valid level-$\alpha$ test.
2. A confidence region $C(X)$ is exactly the set of parameter values not rejected by the corresponding test — this is the test-CI duality.
3. $p$-values depend on the choice of test and should not be confused with the probability that $H_0$ is true. Confidence intervals convey the magnitude and precision of an estimate, not just its significance.

### Next Time (Lecture 15)

- **Generalized likelihood ratio test** (GLRT) and Wilks' theorem: $-2\log\text{LR} \xrightarrow{d} \chi^2$
- The **Wald test** and the **score test**: three asymptotic ways to test $H_0: \theta = \theta_0$
- A unifying picture based on the quadratic approximation to the log-likelihood